# Air Quality from Images — Notebook 2: CNN (transfer learning)
Group 198 · IT401 B.Tech Project-I


In [1]:
# ===================== CONFIG (edit these) =====================
DATA_ROOT  = r"C:\Users\Siddhartha bhansali\OneDrive\Desktop\BtechProject\Air Pollution Image Dataset\Air Pollution Image Dataset"
IMAGE_ROOT = f"{DATA_ROOT}/Combined_Dataset/IND_and_NEP"
CSV_PATH   = f"{DATA_ROOT}/Combined_Dataset/IND_and_Nep_AQI_Dataset.csv"   # note "Nep", not "NEP"
OUTPUT_DIR = f"outputs"
GROUP_COL  = None        # None = auto-detect location/city column (used for leakage-safe split)

IMG_SIZE   = 224
SEED       = 42
SPLIT_MODE = "group"            # "random" or "group"  (group = whole locations held out)
BACKBONE   = "EfficientNetB0"   # "EfficientNetB0" or "MobileNetV2"
BATCH_SIZE = 32
EPOCHS_HEAD, EPOCHS_FT = 8, 20  # phase 1 (frozen backbone), phase 2 (fine-tune)
LR_HEAD, LR_FT = 1e-3, 2e-5
UNFREEZE_LAST = 60              # number of backbone layers to unfreeze in phase 2
USE_CLASS_WEIGHTS = True
STATS_PER_CLASS = 400           # images per class for visual-feature EDA
# ===============================================================

In [2]:
import os, re, json, random, warnings
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,
                             cohen_kappa_score, accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
os.makedirs(OUTPUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
sns.set_theme(style="whitegrid")
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices("GPU"))

c:\Users\Siddhartha bhansali\anaconda3\envs\tfenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TF 2.19.0 | GPUs: []


In [3]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
BASE_COLS = ["path", "folder", "key", "label_name", "label", "label_clean", "group"]

def aqi_rank(name):
    n = name.lower()
    if "very" in n: return 4
    if "hazard" in n or "severe" in n: return 5
    if "sensitive" in n: return 2
    if "unhealthy" in n: return 3
    if "moderate" in n: return 1
    if "good" in n: return 0
    return 99

def clean_name(n):
    return re.sub(r"^[a-z]_", "", n, flags=re.I).replace("_", " ").strip()

def norm_key(s):
    return os.path.splitext(os.path.basename(str(s)))[0].strip().lower()

def build_dataframe():
    paths = [p for p in Path(IMAGE_ROOT).rglob("*") if p.suffix.lower() in IMG_EXTS]
    assert paths, f"No images found under {IMAGE_ROOT}. Fix IMAGE_ROOT."
    df = pd.DataFrame({"path": [str(p) for p in paths], "folder": [p.parent.name for p in paths]})
    df["key"] = df.path.map(norm_key)

    # ---- locate + merge CSV metadata ----
    csv_path = CSV_PATH
    if csv_path is None:
        cands = sorted(Path(DATA_ROOT).rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
        pool = [c for c in cands if "combined" in str(c).lower()] or cands
        csv_path = str(pool[0]) if pool else None
    print("CSV used:", csv_path)
    if csv_path:
        meta = pd.read_csv(csv_path)
        keys = set(df.key)
        best, best_rate = None, 0.0
        for c in meta.columns:
            if meta[c].dtype == object:
                rate = meta[c].map(norm_key).isin(keys).mean()
                if rate > best_rate: best, best_rate = c, rate
        if best and best_rate > 0.5:
            meta = meta.copy(); meta["key"] = meta[best].map(norm_key)
            meta = meta.drop_duplicates("key")
            df = df.merge(meta.drop(columns=[best]), on="key", how="left", suffixes=("", "_csv"))
            print(f"Merged CSV on column '{best}' (match rate {best_rate:.1%})")
        else:
            print("WARNING: could not match any CSV column to image filenames. Metadata skipped.")

    # ---- labels ----
    if df.folder.nunique() >= 2:
        df["label_name"] = df.folder
    else:
        lab = [c for c in df.columns if "class" in c.lower()]
        assert lab, "Images are in one flat folder and CSV has no class column. Send me df.columns."
        df["label_name"] = df[lab[0]].astype(str)
    names = sorted(df.label_name.unique(), key=lambda n: (aqi_rank(n), n))
    df["label"] = df.label_name.map({n: i for i, n in enumerate(names)})
    df["label_clean"] = df.label_name.map(clean_name)

    # ---- group column for leakage-safe split ----
    gcol = GROUP_COL
    if gcol is None:
        for pat in ["location", "site", "station", "city"]:
            m = [c for c in df.columns if pat in c.lower() and c not in BASE_COLS]
            if m: gcol = m[0]; break
    df["group"] = df[gcol].astype(str) if gcol else np.nan
    print("Group column:", gcol)
    return df, names

df, class_names = build_dataframe()
class_clean = [clean_name(n) for n in class_names]
NUM_CLASSES = len(class_names)
print(f"\n{len(df)} images | {NUM_CLASSES} classes: {class_clean}")
df.head()

CSV used: C:\Users\Siddhartha bhansali\OneDrive\Desktop\BtechProject\Air Pollution Image Dataset\Air Pollution Image Dataset/Combined_Dataset/IND_and_Nep_AQI_Dataset.csv
Merged CSV on column 'Filename' (match rate 100.0%)
Group column: Location

12240 images | 6 classes: ['Good', 'Moderate', 'Unhealthy for Sensitive Groups', 'Unhealthy', 'Very Unhealthy', 'Severe']


,path,folder,key,Location,Year,Month,Day,Hour,AQI,PM2.5,PM10,O3,CO,SO2,NO2,AQI_Class,label_name,label,label_clean,group
0,C:\Users\Siddhartha bhansali\OneDrive\Desktop\...,a_Good,bengr_good_2023-02-19-08.30-1-1,Bengaluru,2023,2,19,8:30,41,20.0,36.0,9.0,165.0,3.0,8.0,a_Good,a_Good,0,Good,Bengaluru
1,C:\Users\Siddhartha bhansali\OneDrive\Desktop\...,a_Good,bengr_good_2023-02-19-08.30-1-10,Bengaluru,2023,2,19,8:30,41,20.0,36.0,9.0,165.0,3.0,8.0,a_Good,a_Good,0,Good,Bengaluru
2,C:\Users\Siddhartha bhansali\OneDrive\Desktop\...,a_Good,bengr_good_2023-02-19-08.30-1-11,Bengaluru,2023,2,19,8:30,41,20.0,36.0,9.0,165.0,3.0,8.0,a_Good,a_Good,0,Good,Bengaluru
3,C:\Users\Siddhartha bhansali\OneDrive\Desktop\...,a_Good,bengr_good_2023-02-19-08.30-1-12,Bengaluru,2023,2,19,8:30,41,20.0,36.0,9.0,165.0,3.0,8.0,a_Good,a_Good,0,Good,Bengaluru
4,C:\Users\Siddhartha bhansali\OneDrive\Desktop\...,a_Good,bengr_good_2023-02-19-08.30-1-13,Bengaluru,2023,2,19,8:30,41,20.0,36.0,9.0,165.0,3.0,8.0,a_Good,a_Good,0,Good,Bengaluru


In [4]:
# 6. Perceptual hashes (aHash) -> near-duplicate / leakage analysis
POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)

def ahash(path):
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if g is None: return None
    g = cv2.resize(g, (8, 8), interpolation=cv2.INTER_AREA)
    return np.packbits((g > g.mean()).flatten()).view(np.uint64)[0]

_h = [ahash(p) for p in tqdm(df.path)]
_ok = np.array([h is not None for h in _h])
if not _ok.all():
    print("Dropping", (~_ok).sum(), "unreadable images")
    df = df[_ok].reset_index(drop=True)
hashes = np.array([h for h in _h if h is not None], dtype=np.uint64)
print("Exact-hash duplicate images:", len(hashes) - len(np.unique(hashes)))

def min_hamming(query, ref, chunk=200):
    out = []
    for i in range(0, len(query), chunk):
        x = query[i:i + chunk, None] ^ ref[None, :]
        d = POP[np.ascontiguousarray(x).view(np.uint8).reshape(*x.shape, 8)].sum(-1)
        out.append(d.min(1))
    return np.concatenate(out)

100%|██████████| 12240/12240 [02:07<00:00, 95.92it/s]  

Exact-hash duplicate images: 4810


## PART 2 — CNN (transfer learning, 2-phase training)

In [5]:
# 7. Train / val / test split
def make_splits(df, mode):
    idx, y = np.arange(len(df)), df.label.values
    if mode == "group" and df.group.notna().all() and df.group.nunique() >= 6:
        sgkf = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
        folds = [te for _, te in sgkf.split(idx, y, df.group.values)]
        return np.concatenate(folds[2:]), folds[1], folds[0], "group"
    if mode == "group": print("WARNING: no usable group column -> falling back to random split")
    tr, tmp = train_test_split(idx, test_size=0.3, stratify=y, random_state=SEED)
    va, te = train_test_split(tmp, test_size=0.5, stratify=y[tmp], random_state=SEED)
    return tr, va, te, "random"

tr_idx, va_idx, te_idx, EFFECTIVE_SPLIT = make_splits(df, SPLIT_MODE)
print(f"Split mode: {EFFECTIVE_SPLIT} | train {len(tr_idx)} | val {len(va_idx)} | test {len(te_idx)}")
print(pd.DataFrame({"train": np.bincount(df.label.values[tr_idx], minlength=NUM_CLASSES),
                    "val":   np.bincount(df.label.values[va_idx], minlength=NUM_CLASSES),
                    "test":  np.bincount(df.label.values[te_idx], minlength=NUM_CLASSES)}, index=class_clean))
if EFFECTIVE_SPLIT == "group":
    print("Locations in test:", sorted(set(df.group.values[te_idx])))

# Leakage evidence: how many test images have a near-identical image in train?
d = min_hamming(hashes[te_idx], hashes[tr_idx])
print(f"\nTest images with a near-duplicate in train (aHash Hamming <= 6/64): {(d <= 6).mean():.1%}")

Split mode: group | train 9673 | val 1926 | test 641
                                train  val  test
Good                              919  622     0
Moderate                          611  962     0
Unhealthy for Sensitive Groups   2521  342     0
Unhealthy                        2622    0     0
Very Unhealthy                   1553    0   641
Severe                           1447    0     0
Locations in test: ['New Ind Town, Faridabad']

Test images with a near-duplicate in train (aHash Hamming <= 6/64): 98.3%


In [6]:
# 8. tf.data pipelines
AUTOTUNE = tf.data.AUTOTUNE

def load_img(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img.set_shape((IMG_SIZE, IMG_SIZE, 3))
    return img, label

def make_ds(idx, training):
    ds = tf.data.Dataset.from_tensor_slices((df.path.values[idx], df.label.values[idx].astype("int32")))
    if training: ds = ds.shuffle(len(idx), seed=SEED, reshuffle_each_iteration=True)
    return ds.map(load_img, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds, val_ds, test_ds = make_ds(tr_idx, True), make_ds(va_idx, False), make_ds(te_idx, False)

In [7]:
# 9. Model
# Augmentation is deliberately geometric-only: brightness/contrast jitter would destroy the haze cue we want to learn.
augment = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
    layers.RandomZoom(0.10),
    layers.RandomTranslation(0.05, 0.05),
], name="augment")

def build_model(backbone, n_classes):
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    x = augment(inp)
    if backbone == "EfficientNetB0":
        base = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
    elif backbone == "MobileNetV2":
        base = keras.applications.MobileNetV2(include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3))
        x = layers.Rescaling(1 / 127.5, offset=-1)(x)
    else:
        raise ValueError(backbone)
    base.trainable = False
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(n_classes, activation="softmax")(x)
    return keras.Model(inp, out), base

model, base = build_model(BACKBONE, NUM_CLASSES)
model.compile(optimizer=keras.optimizers.Adam(LR_HEAD), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print(f"{BACKBONE}: {model.count_params():,} total params")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
EfficientNetB0: 4,384,169 total params


In [8]:
# 10. Train: phase 1 (head only) -> phase 2 (fine-tune top layers)
cw = None
if USE_CLASS_WEIGHTS:
    y_tr = df.label.values[tr_idx]; cls = np.unique(y_tr)
    w = compute_class_weight("balanced", classes=cls, y=y_tr)
    cw = {int(c): float(wi ** 0.5) for c, wi in zip(cls, w)}   # sqrt-damped to avoid over-weighting tiny classes
    print("Class weights:", {class_clean[k]: round(v, 2) for k, v in cw.items()})

def cbs(patience):
    return [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7)]

print("=== Phase 1: training classifier head ===")
h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, class_weight=cw, callbacks=cbs(4))

print("=== Phase 2: fine-tuning ===")
base.trainable = True
for l in base.layers[:-UNFREEZE_LAST]: l.trainable = False
for l in base.layers:
    if isinstance(l, layers.BatchNormalization): l.trainable = False   # keep BN stats frozen
model.compile(optimizer=keras.optimizers.Adam(LR_FT), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, class_weight=cw, callbacks=cbs(6))

hist = {k: h1.history[k] + h2.history[k] for k in ["accuracy", "val_accuracy", "loss", "val_loss"]}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, (k, t) in zip(ax, [("accuracy", "Accuracy"), ("loss", "Loss")]):
    a.plot(hist[k], label="train"); a.plot(hist["val_" + k], label="val")
    a.axvline(len(h1.history["loss"]) - 1, ls="--", c="gray", label="fine-tune starts"); a.set_title(t); a.legend()
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/training_curves_{BACKBONE}_{EFFECTIVE_SPLIT}.png", dpi=130); plt.show()

model.save(f"{OUTPUT_DIR}/{BACKBONE}_{EFFECTIVE_SPLIT}.keras")

Class weights: {'Good': 1.32, 'Moderate': 1.62, 'Unhealthy for Sensitive Groups': 0.8, 'Unhealthy': 0.78, 'Very Unhealthy': 1.02, 'Severe': 1.06}
=== Phase 1: training classifier head ===
Epoch 1/8
 17/303 ━━━━━━━━━━━━━━━━━━━━ 2:26 511ms/step - accuracy: 0.3103 - loss: 2.2527

KeyboardInterrupt: 

In [ ]:
# 11. Evaluation on the held-out test set
probs = model.predict(test_ds, verbose=0)
pred, y_true = probs.argmax(1), df.label.values[te_idx]
labels_all = list(range(NUM_CLASSES))

res = dict(backbone=BACKBONE, split=EFFECTIVE_SPLIT, n_test=int(len(y_true)),
           accuracy=accuracy_score(y_true, pred),
           macro_f1=f1_score(y_true, pred, average="macro", labels=np.unique(y_true), zero_division=0),
           within_1_class=float((np.abs(pred - y_true) <= 1).mean()),
           qwk=cohen_kappa_score(y_true, pred, weights="quadratic"),
           mae_class_idx=float(np.abs(pred - y_true).mean()),
           near_dup_in_train=float((d <= 6).mean()))
print(json.dumps({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()}, indent=2))
print("\n", classification_report(y_true, pred, labels=labels_all, target_names=class_clean, zero_division=0))

cm = confusion_matrix(y_true, pred, labels=labels_all)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_clean, yticklabels=class_clean, ax=ax[0]); ax[0].set_title("Confusion matrix (counts)")
sns.heatmap(cmn, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_clean, yticklabels=class_clean, ax=ax[1]); ax[1].set_title("Row-normalised")
for a in ax: a.set_xlabel("Predicted"); a.set_ylabel("True"); a.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.savefig(f"{OUTPUT_DIR}/confusion_{BACKBONE}_{EFFECTIVE_SPLIT}.png", dpi=130); plt.show()

# append to results table (re-run with other SPLIT_MODE / BACKBONE to build the comparison)
rp = f"{OUTPUT_DIR}/results.json"
allres = json.load(open(rp)) if os.path.exists(rp) else []
allres.append(res); json.dump(allres, open(rp, "w"), indent=2)
pd.DataFrame(allres).round(4)